# Proteomics
Extract timepoints from dataset and create two new datasets from it
Creates new *mid_processing_datasets* folder and parquet files during runtime
Drops high corellating features and saves final output to *cleaned_datasets*

In [12]:
from src.common_imports import *
from src.data_audit import data_audit
from src.somascan import *
from src.save_wrapper import save_report
from src.extraction import extract_dtypes
from src.check_data import *
from src.normalization import *
from src.low_variance_report import low_variance_report
from src.featsel import run_correlation_selection



In [13]:
data = load()

Loading somascan...
Done loading!


In [14]:
print("Indexing data...")
df_exp = data['df_expMatrix']
df_samp = data['df_sampMatrix']
print("Done!")

Indexing data...
Done!


# Data and low variance reports
Shows an overview of the following
data_audit:
-   Dataframe shape ( rows x columns)
-   Duplicates
-   Missing Values
-   Numeric Features
-   Outliers by IQR and 3σ(68-95-99.7)
-   Categorical Consistency
-   Cleaning Summary

low_variance_report:
-   Total numeric features
-   predefined low_variance_threshold
-   low variance features found

In [15]:
# data analysis of raw data
save_report(data_audit, df_exp, "expression matrix")
save_report(data_audit,df_samp, "sample matrix")
save_report(low_variance_report, df_exp, "expression matrix")
save_report(low_variance_report, df_samp, "sample matrix")

Saved: ../../reports/expression matrix_data_audit.txt
Saved: ../../reports/sample matrix_data_audit.txt
Saved: ../../reports/expression matrix_low_variance_report.txt
Saved: ../../reports/sample matrix_low_variance_report.txt


# Extracting and saving
After taking a look at the data we extract them by timepoint
and split into two new datasets
additionally we create a pseudo glossary for if one wants to lookup the gene

In [16]:
# save splits
splits = reshape_and_split(df_exp, df_samp)

# save splits
save_splits(splits)

# create glossary
create_gene_lookup(df_exp)

Timepoints found: ['6month', 'Baseline']
  6month: (97, 1311)
  Baseline: (140, 1311)
Saved: ../../mid_processing_datasets/expression_matrix_6month.parquet
Saved: ../../mid_processing_datasets/expression_matrix_baseline.parquet
Saved: ../../mid_processing_datasets/gene_lookup.parquet


,EntrezGeneSymbol,EntrezGeneID,Target,TargetFullName,UniProt
SeqId,,,,,
2597-8_3,VEGFA,7422,VEGF,Vascular endothelial growth factor A,P15692
2654-19_1,TNFRSF1A,7132,TNF sR-I,Tumor necrosis factor receptor superfamily mem...,P19438
2788-55_1,MMP3,4314,MMP-3,Stromelysin-1,P08254
2967-8_1,VCAM1,7412,VCAM-1,Vascular cell adhesion protein 1,P19320
3046-31_1,RETN,56729,resistin,Resistin,Q9HD89
...,...,...,...,...,...
9211-19_3,SERPINF1,5176,PEDF,Pigment epithelium-derived factor,P36955
9212-22_3,CTSF,8722,CATF,Cathepsin F,Q9UBX1
9213-24_3,FTCD,10841,FTCD,Formimidoyltransferase-cyclodeaminase,O95954


# Verifiying
After saving the files we verify it by pulling from the new source
and running the data_audit and low_variance_report on them

In [17]:
# pulling saved data
df_exp_bl = pd.read_parquet("../../mid_processing_datasets/expression_matrix_baseline.parquet")
df_exp_6m = pd.read_parquet("../../mid_processing_datasets/expression_matrix_6month.parquet")
print("done reading")

done reading


In [18]:
# running data audit on newly formed data
save_report(data_audit, df_exp_bl, "baseline expression matrix")
save_report(data_audit, df_exp_6m, "6 months expression matrix")
save_report(low_variance_report,df_exp_bl, "baseline expression matrix")
save_report(low_variance_report, df_exp_6m, "6 months expression matrix")

Saved: ../../reports/baseline expression matrix_data_audit.txt
Saved: ../../reports/6 months expression matrix_data_audit.txt
Saved: ../../reports/baseline expression matrix_low_variance_report.txt
Saved: ../../reports/6 months expression matrix_low_variance_report.txt


In [19]:
# further sample verificiation
display(df_exp_bl.head())
display(df_exp_6m.head())

SeqId,Patient_ID,2597-8_3,2654-19_1,2788-55_1,2967-8_1,3046-31_1,4336-2_1,4337-49_2,4673-13_2,4867-15_2,...,9199-6_3,9201-13_3,9202-309_3,9204-33_3,9207-60_3,9211-19_3,9212-22_3,9213-24_3,9215-117_3,9216-100_3
SampleId,,,,,,,,,,,,,,,,,,,,,
TAC1145_BL,TAC1145,16301.7,1628.1,689.4,12386.8,1732.7,2461.5,146207.9,559.6,1556.6,...,7101.8,2950.0,1894.1,636.1,3068.7,63197.8,2521.2,7522.2,1114.9,4627.6
TAC1000_BL,TAC1000,10421.1,1220.8,426.4,11669.8,1042.4,1676.3,157654.1,787.4,512.5,...,5669.6,7869.2,7685.5,489.2,1507.6,57489.0,2201.4,6938.8,804.8,4887.5
TAC1058_BL,TAC1058,9352.7,1337.1,668.6,15387.8,2402.8,1377.1,142311.8,225.7,499.3,...,6163.6,2828.3,2822.8,645.4,2429.1,53653.8,1937.3,14913.5,1242.0,4860.3
TAC1150_BL,TAC1150,11279.3,1355.2,390.8,12320.6,1350.4,216.0,41350.3,266.6,541.4,...,5003.9,3559.3,13755.8,987.8,5549.8,53542.5,2049.0,24307.1,683.9,4903.5
TAC1174_BL,TAC1174,10622.2,1062.2,671.9,16307.8,1323.5,418.5,87174.8,232.8,429.1,...,8339.7,3676.7,1498.4,779.9,2497.6,55372.4,2975.7,14636.7,925.1,7282.2


SeqId,Patient_ID,2597-8_3,2654-19_1,2788-55_1,2967-8_1,3046-31_1,4336-2_1,4337-49_2,4673-13_2,4867-15_2,...,9199-6_3,9201-13_3,9202-309_3,9204-33_3,9207-60_3,9211-19_3,9212-22_3,9213-24_3,9215-117_3,9216-100_3
SampleId,,,,,,,,,,,,,,,,,,,,,
TAC1075_M6,TAC1075,11993.7,1358.4,585.6,14119.3,997.0,2150.9,81794.4,512.9,633.8,...,3776.1,2515.1,1780.4,1248.7,1799.0,55797.4,2078.4,9340.3,1061.8,5881.7
TAC1148_M6,TAC1148,11000.8,1087.1,602.3,13485.6,1186.2,342.0,16741.7,222.5,417.4,...,4251.7,3025.8,1769.1,500.5,3253.0,59882.3,2515.2,10679.1,913.7,5373.4
TAC1179_M6,TAC1179,15116.4,1205.6,382.9,13505.8,1155.5,413.7,116629.2,383.2,1135.1,...,10580.8,5813.3,3905.4,568.0,1714.8,61199.4,1863.4,3418.3,780.7,4142.3
TAC1225_M6,TAC1225,12873.1,2062.8,1243.5,20003.1,1864.4,301.3,119197.3,274.8,398.6,...,6877.5,3461.2,3426.8,668.2,2726.7,72002.7,2514.8,13206.6,699.5,5775.7
TAC1209_M6,TAC1209,10796.4,1178.1,1136.9,15152.3,1364.6,863.3,131055.7,273.4,435.8,...,4859.8,3049.0,1912.6,515.5,2130.7,56690.5,2221.6,12061.3,887.9,4544.7


In [20]:
extract_dtypes(df_exp_bl, verbose=True)

Schema of total values 
(column, row)
Absolute Values: (140, 1311)
['Patient_ID', '2597-8_3', '2654-19_1', '2788-55_1', '2967-8_1', '3046-31_1', '4336-2_1', '4337-49_2', '4673-13_2', '4867-15_2', '4924-32_1', '5509-7_3', '8484-24_3', '10336-3_3', '10337-83_3', '10339-48_3', '10342-55_3', '10344-334_3', '10346-5_3', '10351-51_3', '10356-21_3', '10358-33_3', '10361-25_3', '10362-35_3', '10363-13_3', '10364-6_3', '10365-132_3', '10366-11_3', '10367-62_3', '10370-21_3', '10372-18_3', '10990-21_3', '11067-13_3', '11071-1_3', '11081-1_3', '11089-7_3', '11094-104_3', '11096-57_3', '11098-1_3', '11101-18_3', '11102-22_3', '11103-24_3', '11104-13_3', '11105-171_3', '11510-31_3', '11513-92_3', '11514-196_3', '11516-7_3', '12060-28_3', '13088-397_3', '13089-6_3', '13090-17_3', '13093-6_3', '13094-75_3', '13095-51_3', '13097-11_3', '13098-93_3', '13101-60_3', '13102-1_3', '13103-125_3', '13104-32_3', '13105-7_3', '13107-9_3', '13109-82_3', '13111-79_3', '13112-179_3', '13113-7_3', '13114-50_3', '1

(SeqId      Patient_ID  2597-8_3  2654-19_1  2788-55_1  2967-8_1  3046-31_1  \
 SampleId                                                                     
 TAC1145_BL    TAC1145   16301.7     1628.1      689.4   12386.8     1732.7   
 TAC1000_BL    TAC1000   10421.1     1220.8      426.4   11669.8     1042.4   
 TAC1058_BL    TAC1058    9352.7     1337.1      668.6   15387.8     2402.8   
 TAC1150_BL    TAC1150   11279.3     1355.2      390.8   12320.6     1350.4   
 TAC1174_BL    TAC1174   10622.2     1062.2      671.9   16307.8     1323.5   
 ...               ...       ...        ...        ...       ...        ...   
 TAC1042_BL    TAC1042    9384.1      943.9      814.4   10377.0     2250.5   
 TAC1232_BL    TAC1232   11154.9     1281.2     1889.8   17802.6     1633.2   
 TAC1082_BL    TAC1082    6652.2     1636.0     1102.0   11790.8      979.0   
 TAC1179_BL    TAC1179   16324.1     1614.3      601.3   15779.1     1683.5   
 VAC2038_BL    VAC2038    7696.6     1052.6      740

In [21]:
#show_sheet_overview(df_exp_bl)
#show_dataset_structure(df_exp_bl)
inspect_omics_dataset(df_exp_bl)


===== OMICS DATASET =====

SAMPLE:


SeqId,Patient_ID,2597-8_3,2654-19_1,2788-55_1,2967-8_1,3046-31_1,4336-2_1,4337-49_2,4673-13_2,4867-15_2,...,9199-6_3,9201-13_3,9202-309_3,9204-33_3,9207-60_3,9211-19_3,9212-22_3,9213-24_3,9215-117_3,9216-100_3
SampleId,,,,,,,,,,,,,,,,,,,,,
TAC1251_BL,TAC1251,10883.7,1111.1,806.9,16868.5,989.0,455.8,135501.8,235.8,628.1,...,6355.1,3812.8,2714.1,556.2,3029.5,67230.2,3073.5,49499.8,1139.8,3983.0
TAC1055_BL,TAC1055,11341.1,1829.0,4069.6,15120.7,2095.8,928.3,94405.1,586.7,740.4,...,4501.6,4654.4,8973.6,1167.9,1751.1,53744.0,3349.7,7259.6,799.0,6012.9
TAC1212_BL,TAC1212,12009.3,1322.1,560.3,20374.6,2371.2,3856.2,125999.2,697.9,573.0,...,5764.7,3566.5,10733.2,530.0,1787.5,57312.8,1881.9,6559.9,1006.0,5415.8
TAC1239_BL,TAC1239,13333.0,2748.7,755.7,32853.2,2114.2,360.6,116396.6,387.3,513.0,...,4450.1,3871.3,5100.1,672.9,3175.9,48853.1,2451.6,29919.3,1381.4,8790.1
TAC1137_BL,TAC1137,9810.1,1406.8,811.4,9458.9,912.9,354.7,110819.5,252.8,415.2,...,5620.1,2276.4,21120.8,658.6,3028.2,57417.5,2433.1,65276.9,912.3,5497.7



SHAPE:
(140, 1311)

DTYPES:
SeqId
Patient_ID        str
2597-8_3      float64
2654-19_1     float64
2788-55_1     float64
2967-8_1      float64
               ...   
9211-19_3     float64
9212-22_3     float64
9213-24_3     float64
9215-117_3    float64
9216-100_3    float64
Length: 1311, dtype: object

MISSING VALUES (Top 10):
SeqId
Patient_ID    0
2597-8_3      0
2654-19_1     0
2788-55_1     0
2967-8_1      0
3046-31_1     0
4336-2_1      0
4337-49_2     0
4673-13_2     0
4867-15_2     0
dtype: int64

DTYPES (10 random samples):
SeqId
2418-55_9     float64
8470-213_3    float64
10356-21_3    float64
2748-3_2      float64
2658-27_1     float64
6653-58_3     float64
2666-53_2     float64
4254-6_2      float64
4469-78_2     float64
4155-3_2      float64
dtype: object

SHORT NUMERIC SUMMARY (random 10 samples):


,count,mean,std,min,25%,50%,75%,max
SeqId,,,,,,,,
11067-13_3,140.0,2925.581429,11331.862452,760.6,1072.700,1429.45,1982.625,134001.0
4355-13_1,140.0,2153.324286,1555.867836,1177.5,1683.250,1898.25,2110.550,18576.8
9169-14_3,140.0,1975.699286,1806.938955,771.4,1269.275,1589.30,2172.575,20101.3
5201-50_4,140.0,793.474286,2128.966711,278.2,403.900,447.00,601.225,23127.4
4556-10_2,140.0,562.979286,129.221445,275.0,494.275,562.60,646.275,1051.5
2937-10_2,140.0,222364.665714,46358.271127,117471.0,187464.950,223824.05,252490.700,331874.8
5903-91_2,140.0,2022.783571,1364.958045,1267.6,1615.600,1829.05,2049.525,13796.3
5018-68_1,140.0,439.700714,207.152830,167.3,298.575,378.15,521.725,1421.8
4254-6_2,140.0,245.360000,312.891190,103.1,161.575,199.55,248.850,3476.5


# Feature Removal
Pairwise high correlation removal (threshold=0.95)
For each correlated pair, the feature with lower variance is dropped
The feature with higher variance is always retained

In [22]:
df_exp_bl_clean, to_keep_bl, to_drop_bl = run_correlation_selection(df=df_exp_bl,
                        name='baseline expression proteomics',
                        threshold=0.95, output_dir='../../reports/feature_selection' )

df_exp_m6_clean, to_keep_m6, to_drop_m6 = run_correlation_selection(df=df_exp_6m,
                        name='6m expression proteomics',
                        threshold=0.95, output_dir='../../reports/feature_selection' )


FEATURE SELECTION: BASELINE EXPRESSION PROTEOMICS  (threshold = 0.95)
Computing correlation matrix for 1310 features (140 samples) …

Results:
  Total numeric features : 1310
  Kept                   : 1230 (93.9%)
  Dropped (redundant)    : 80 (6.1%)

Saving feature lists ...
  Saved: ../../reports/feature_selection\baseline expression proteomics_features_to_keep.txt  (1230 features)
  Saved: ../../reports/feature_selection\baseline expression proteomics_features_to_drop.txt  (80 features)
  Saved: ../../reports/feature_selection\baseline expression proteomics_feature_selection.json

FEATURE SELECTION: 6M EXPRESSION PROTEOMICS  (threshold = 0.95)
Computing correlation matrix for 1310 features (97 samples) …

Results:
  Total numeric features : 1310
  Kept                   : 1232 (94.0%)
  Dropped (redundant)    : 78 (6.0%)

Saving feature lists ...
  Saved: ../../reports/feature_selection\6m expression proteomics_features_to_keep.txt  (1232 features)
  Saved: ../../reports/feature_s

In [ ]:
df_exp_bl_clean.to_parquet('../../cleaned_datasets/exp_bl.parquet')

df_exp_m6_clean.to_parquet('../../cleaned_datasets/exp_m6.parquet')

df_exp_bl_clean.to_csv('../cleaned_datasets/exp_bl.csv')
df_exp_6m.to_csv('../cleaned_datasets/exp_m6.csv')

# Summary
After running this notebook the preprocessing has been done and the data is ready for merge or be analyzed further